# 02 — Pull World Bank WDI + WGI

**Sources:**
- **World Development Indicators (WDI)** — economic, social, demographic, technology indicators  
- **Worldwide Governance Indicators (WGI)** — six governance dimensions  

**Provider:** World Bank  
**Access:** `wbgapi` Python package — no API key required  
**Coverage:** ~217 economies, 1960–present (indicator-dependent)  
**Frequency:** Annual  

## What this notebook does
1. Pulls the WDI indicators used as **structural/economic predictors** in the instability models
2. Pulls the WGI governance indicators (also used as predictors)
3. Writes a wide-format country-year panel for each source to ADLS

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

## Indicators pulled
See DatasetVariableSynthesis.md §1.2 (World Bank WDI) and meta-plan Domains 1 & 2.

In [ ]:
import os
import wbgapi
import pandas as pd
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
START_YEAR        = 1995   # 5-year buffer before 2000 panel start for lag features
END_YEAR          = datetime.utcnow().year
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

print(f"Year range  : {START_YEAR}–{END_YEAR}")
print(f"ADLS target : abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/raw/world_bank/")
print(f"Run date    : {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## WDI indicator definitions

Indicators are grouped by thematic domain to match the meta-plan feature taxonomy.

In [ ]:
WDI_INDICATORS = {
    # --- Economic (Domain 1) ---
    "NY.GDP.PCAP.KD":     "gdp_per_capita_const_usd",
    "NY.GDP.PCAP.PP.KD":  "gdp_per_capita_ppp",
    "NY.GDP.MKTP.KD.ZG":  "gdp_growth_pct",
    "FP.CPI.TOTL.ZG":     "inflation_cpi_pct",
    "SL.UEM.TOTL.ZS":     "unemployment_pct",
    "SI.POV.GINI":        "gini_coefficient",
    "NY.GDP.TOTL.RT.ZS":  "resource_rents_pct_gdp",
    "BX.KLT.DINV.WD.GD.ZS": "fdi_net_inflows_pct_gdp",
    "GC.DOD.TOTL.GD.ZS":  "central_govt_debt_pct_gdp",
    "BN.CAB.XOKA.GD.ZS":  "current_account_balance_pct_gdp",
    "NE.TRD.GNFS.ZS":     "trade_openness_pct_gdp",
    # --- Demographic / Social (Domain 1 + 3) ---
    "SP.POP.TOTL":        "population_total",
    "SP.POP.GROW":        "population_growth_pct",
    "SP.DYN.IMRT.IN":     "infant_mortality_per_1000",
    "SP.URB.TOTL.IN.ZS":  "urban_population_pct",
    "SP.POP.0014.TO.ZS":  "pop_pct_0_14",
    "SP.POP.1564.TO.ZS":  "pop_pct_15_64",
    "SH.DYN.MORT":        "under5_mortality_per_1000",
    "SE.ADT.LITR.ZS":     "adult_literacy_pct",
    "SH.XPD.CHEX.GD.ZS":  "health_expenditure_pct_gdp",
    # --- Technology / Information (Domain 5) ---
    "IT.CEL.SETS.P2":     "mobile_subscriptions_per_100",
    "IT.NET.USER.ZS":     "internet_users_pct",
    "IT.NET.BBND.P2":     "broadband_subs_per_100",
    # --- Military (for SIPRI cross-check) ---
    "MS.MIL.XPND.GD.ZS":  "military_expenditure_pct_gdp",
    "MS.MIL.TOTL.P1":     "armed_forces_personnel_total",
}

print(f"Pulling {len(WDI_INDICATORS)} WDI indicators")

## Pull WDI data

In [ ]:
print("Pulling WDI from World Bank API...")
df_wdi_raw = wbgapi.data.DataFrame(
    series=list(WDI_INDICATORS.keys()),
    time=range(START_YEAR, END_YEAR + 1),
    labels=True,   # include country and indicator labels
    numericTimeKeys=True,
).reset_index()

print(f"Raw shape: {df_wdi_raw.shape}")
df_wdi_raw.head(3)

## Reshape to country-year panel (wide format)

`wbgapi` returns a multi-level DataFrame; we pivot to one row per country-year.

In [ ]:
# wbgapi returns columns = indicator codes, index = (economy, time)
# Reset to flat and rename
df_wdi = df_wdi_raw.copy()

# Standardise index column names produced by wbgapi
rename_index = {}
for col in df_wdi.columns:
    if col.lower() in ("economy", "economycode"):
        rename_index[col] = "iso3"
    elif col.lower() in ("time", "year"):
        rename_index[col] = "year"
    elif col.lower() in ("economy label", "economylabel", "country"):
        rename_index[col] = "country_name"
df_wdi.rename(columns=rename_index, inplace=True)

# Rename indicator codes to friendly names
df_wdi.rename(columns=WDI_INDICATORS, inplace=True)

# Keep only sovereign states (drop World Bank aggregates like 'WLD', 'LIC', etc.)
# ISO3 codes for aggregates are typically 3 digits or start with a number
if "iso3" in df_wdi.columns:
    df_wdi = df_wdi[df_wdi["iso3"].str.match(r"^[A-Z]{3}$", na=False)]

df_wdi["year"] = df_wdi["year"].astype(int)

print(f"WDI panel shape: {df_wdi.shape}")
print(f"Countries      : {df_wdi['iso3'].nunique() if 'iso3' in df_wdi.columns else 'n/a'}")
print(f"Years          : {df_wdi['year'].min()}–{df_wdi['year'].max()}")
df_wdi.head(3)

## WGI indicator definitions

In [ ]:
# WGI indicators live in a separate World Bank source (source ID 3)
WGI_INDICATORS = {
    "CC.EST":  "wgi_control_of_corruption",
    "GE.EST":  "wgi_govt_effectiveness",
    "PV.EST":  "wgi_political_stability",
    "RL.EST":  "wgi_rule_of_law",
    "RQ.EST":  "wgi_regulatory_quality",
    "VA.EST":  "wgi_voice_accountability",
}
print(f"Pulling {len(WGI_INDICATORS)} WGI indicators")

## Pull WGI data

In [ ]:
print("Pulling WGI from World Bank API...")
df_wgi_raw = wbgapi.data.DataFrame(
    series=list(WGI_INDICATORS.keys()),
    time=range(START_YEAR, END_YEAR + 1),
    labels=True,
    numericTimeKeys=True,
    db=3,   # WGI is World Bank source database 3
).reset_index()

print(f"Raw shape: {df_wgi_raw.shape}")
df_wgi_raw.head(3)

In [ ]:
df_wgi = df_wgi_raw.copy()

rename_index = {}
for col in df_wgi.columns:
    if col.lower() in ("economy", "economycode"):
        rename_index[col] = "iso3"
    elif col.lower() in ("time", "year"):
        rename_index[col] = "year"
    elif col.lower() in ("economy label", "economylabel", "country"):
        rename_index[col] = "country_name"
df_wgi.rename(columns=rename_index, inplace=True)
df_wgi.rename(columns=WGI_INDICATORS, inplace=True)

if "iso3" in df_wgi.columns:
    df_wgi = df_wgi[df_wgi["iso3"].str.match(r"^[A-Z]{3}$", na=False)]

df_wgi["year"] = df_wgi["year"].astype(int)

print(f"WGI panel shape: {df_wgi.shape}")
print(f"Countries      : {df_wgi['iso3'].nunique() if 'iso3' in df_wgi.columns else 'n/a'}")
df_wgi.head(3)

## Write both panels to ADLS

In [ ]:
write_parquet(df_wdi, f"raw/world_bank/wdi/{RUN_DATE}/wdi_panel.parquet")
write_parquet(df_wgi, f"raw/world_bank/wgi/{RUN_DATE}/wgi_panel.parquet")

## Summary

In [ ]:
for name, df in [("WDI", df_wdi), ("WGI", df_wgi)]:
    non_null = df.drop(columns=["iso3", "year"], errors="ignore").notna().mean().mean()
    print(f"{name}: {len(df):,} rows | {df['iso3'].nunique()} countries "
          f"| {df['year'].min()}–{df['year'].max()} "
          f"| {non_null:.0%} non-null indicator values")